# Imports de librerias

In [0]:

from pyspark.sql.functions import col, lower, round

# Lectura de la Tabla Bronce olist_order_payments

In [0]:
df = spark.table("`catalog_brazilian-e-commerce`.bronze.olist_order_payments_dataset")

In [0]:
df.display()

#Transformaciones

In [0]:


df = (
    df
    # tipo de datos
    .withColumn("payment_sequential", col("payment_sequential").cast("int"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
    .withColumn("payment_value", col("payment_value").cast("double"))

    # limpieza de duplicados o nulos
    .dropDuplicates()
    .dropna(subset=["order_id", "payment_type", "payment_value"])

    # normalización a minuscula
    .withColumn("payment_type", lower(col("payment_type")))

    # filtro de valores mayores o iguales a 0
    .filter(col("payment_value") >= 0)

    # que quede con 2 decimales
    .withColumn(
        "payment_value",
        round(col("payment_value"), 2)
    )
)

In [0]:
df.display()

#Crear la tabla Silver de olist_order_payments

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.silver.olist_order_payments")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.silver.olist_order_payments;